# THEMAP — 5-minute quick tour

<a href="https://colab.research.google.com/github/HFooladi/THEMAP/blob/main/notebooks/colab/01_quick_tour.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**THEMAP** computes distances between *chemical datasets* — not between individual molecules. This is the operation you need when:

- you have a target prediction task and want to pick the most relevant source dataset for transfer learning,
- you want to score how *hard* a new bioactivity task is going to be before training a model,
- you want to map out the landscape of your dataset collection (which tasks resemble each other).

This notebook takes about 5 minutes end-to-end on a free CPU Colab runtime. We will:

1. install THEMAP,
2. download 3 source + 2 target ChEMBL datasets (~400 KB),
3. peek inside one dataset,
4. compute a 2×3 distance matrix with ECFP fingerprints + Euclidean distance,
5. visualise the matrix and pick the nearest source for each target.

No GPU and no heavy neural featurizers are used. If you like what you see, head to [`02_api_deep_dive.ipynb`](02_api_deep_dive.ipynb) for the programmatic API.

## 1. Install

We install `themap` directly from GitHub (the package isn't on PyPI yet) plus `molfeat` (fingerprints) and `matplotlib`/`seaborn` (plotting). `rdkit` is pulled in automatically as a `themap` core dependency. On a fresh Colab runtime this takes ~2–3 minutes.

In [ ]:
%pip install -q git+https://github.com/HFooladi/THEMAP.git molfeat matplotlib seaborn

## 2. Download a small sample dataset

We pull 5 ChEMBL assay files straight from the THEMAP GitHub repo and arrange them in the layout THEMAP expects: `datasets/train/` for source tasks, `datasets/test/` for target tasks. Each `*.jsonl.gz` is one assay, with `SMILES` + binary `Property` labels.

In [ ]:
import os
import urllib.request

BASE = "https://raw.githubusercontent.com/HFooladi/THEMAP/main/datasets"
SAMPLES = {
    "train": ["CHEMBL3371729", "CHEMBL894522", "CHEMBL4224224"],
    "test": ["CHEMBL2219236", "CHEMBL2219358"],
}

for fold, ids in SAMPLES.items():
    os.makedirs(f"datasets/{fold}", exist_ok=True)
    for cid in ids:
        dst = f"datasets/{fold}/{cid}.jsonl.gz"
        if not os.path.exists(dst):
            urllib.request.urlretrieve(f"{BASE}/{fold}/{cid}.jsonl.gz", dst)
        print(f"  {dst}  ({os.path.getsize(dst) / 1024:.1f} KB)")

## 3. Peek inside one dataset

`MoleculeDataset.load_from_file` reads one `.jsonl.gz` into the in-memory representation THEMAP uses internally: a list of SMILES + a numpy array of binary labels.

In [ ]:
from themap import MoleculeDataset

ds = MoleculeDataset.load_from_file("datasets/test/CHEMBL2219236.jsonl.gz")
print(f"task_id        : {ds.task_id}")
print(f"#molecules     : {len(ds)}")
print(f"positive ratio : {ds.positive_ratio:.2%}")
print("first 5 SMILES :")
for s, y in zip(ds.smiles_list[:5], ds.labels[:5]):
    print(f"  [{y}] {s}")

## 4. Compute the dataset distance matrix

`quick_distance` is the one-call entry point: it discovers every file in `train/` and `test/`, featurizes them with the chosen fingerprint, and returns an N×M dict of `target_id -> source_id -> distance`. With `ecfp` + `euclidean` this runs in seconds on CPU.

In [ ]:
from themap import quick_distance

results = quick_distance(
    data_dir="datasets",
    output_dir="output",
    molecule_featurizer="ecfp",
    molecule_method="euclidean",
    n_jobs=2,
)
results["molecule"]

## 5. Visualise and interpret

`quick_distance` also dropped a CSV at `output/molecule_distances.csv` — **rows are targets, columns are sources**. A heatmap is the fastest way to see structure, and `idxmin()` along each row tells us the nearest source for each target (the natural pick for transfer learning).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

distances = pd.read_csv("output/molecule_distances.csv", index_col=0)
distances.index.name, distances.columns.name = "target", "source"
print("Distance matrix (rows = targets, cols = sources):")
print(distances.round(3))

fig, ax = plt.subplots(figsize=(5, 3))
sns.heatmap(distances, annot=True, fmt=".2f", cmap="viridis_r", ax=ax)
ax.set_title("ECFP + Euclidean distance (lower = more similar)")
plt.tight_layout()
plt.show()

print("\nClosest source for each target:")
for tgt in distances.index:
    src = distances.loc[tgt].idxmin()
    print(f"  {tgt}  <-  {src}   (d = {distances.loc[tgt].min():.3f})")

## Where to go next

- **[`02_api_deep_dive.ipynb`](02_api_deep_dive.ipynb)** — the programmatic API: `DatasetLoader`, `MoleculeFeaturizer`, `DatasetDistance` directly, YAML pipeline configs, a UMAP task landscape, and an optional GPU/OTDD cell.
- **CLI** — everything you just did is also available from a terminal: `themap quick datasets/ -f ecfp -m euclidean -o output/`.
- **Docs** — [hfooladi.github.io/THEMAP](https://hfooladi.github.io/THEMAP/).
- **Paper** — [Fooladi et al., *J. Chem. Inf. Model.* 64(10), 4031–4046 (2024)](https://doi.org/10.1021/acs.jcim.4c00160).